# Day 4 · 03 · Merge and Join

`merge` is pandas' SQL join. The mechanics are simple; the skill is in **checking** that a merge did what you expected.

**In this notebook**
- `merge` and the `how` options
- `indicator`: where did each row come from?
- Keys with different names, and a self-merge
- `validate`: state the relationship you expect
- Fan-out: the merge that inflates your totals
- Keys that do not match: dtype and formatting problems
- `join` on the index and `concat`
- `merge_asof`: the latest value as of a date
- Cross joins

**Dataset:** the retail CSV files and `orders_raw.csv`

In [1]:
from pathlib import Path

import pandas as pd

DATA = Path("../datasets").resolve()

orders = pd.read_csv(DATA / "orders.csv", parse_dates=["order_date"])
items = pd.read_csv(DATA / "order_items.csv")
products = pd.read_csv(DATA / "products.csv")
customers = pd.read_csv(DATA / "customers.csv")
staff = pd.read_csv(DATA / "staff.csv")

items["revenue"] = items["quantity"] * items["unit_price"] * (1 - items["discount"])

## 1. `merge` basics
`left.merge(right, on="key", how="inner")` matches rows whose key values are equal.

| `how` | Rows kept | SQL |
| --- | --- | --- |
| `"inner"` (default) | keys present in both | `INNER JOIN` |
| `"left"` | every left row | `LEFT JOIN` |
| `"right"` | every right row | `RIGHT JOIN` |
| `"outer"` | every row from both | `FULL OUTER JOIN` |
| `"cross"` | every combination | `CROSS JOIN` |

Always compare the row count before and after.

In [2]:
with_customer = orders.merge(customers[["customer_id", "name", "country"]], on="customer_id")

print("orders:", len(orders), "| after merge:", len(with_customer))
with_customer.head(3)

orders: 2000 | after merge: 2000


,order_id,customer_id,order_date,status,channel,staff_id,name,country
0,10001,C0176,2024-01-01,completed,mobile,NaN,Chloe Brown,Spain
1,10002,C0200,2024-01-01,returned,mobile,NaN,Chloe Schmidt,India
2,10003,C0203,2024-01-01,completed,web,NaN,Sofia Miller,UK


## 2. `indicator`: see which side each row came from
`indicator=True` adds a `_merge` column with `both`, `left_only` or `right_only`. It turns a merge into a reconciliation report.

Rows that are `left_only` form the **anti-join**: here, customers who never ordered.

In [3]:
ordering_customers = orders[["customer_id"]].drop_duplicates()
check = customers.merge(ordering_customers, on="customer_id", how="left", indicator=True)

print(check["_merge"].value_counts().to_string())

never_ordered = check.loc[check["_merge"] == "left_only", ["customer_id", "name", "country"]]
never_ordered.head(3)

_merge
both          205
left_only      45
right_only      0


,customer_id,name,country
3,C0004,Lucia Murphy,Spain
8,C0009,Jonas Lopez,India
11,C0012,Grace Taylor,UK


In [4]:
# An outer merge reconciles both directions at once: products never sold, and sold ids missing from the catalogue.
units_sold = items.groupby("product_id", as_index=False)["quantity"].sum()
reconcile = products.merge(units_sold, on="product_id", how="outer", indicator=True)

print(reconcile["_merge"].value_counts().to_string())
reconcile.loc[reconcile["_merge"] != "both", ["product_id", "product_name", "quantity", "_merge"]]

_merge
both          37
left_only      3
right_only     0


,product_id,product_name,quantity,_merge
37,P038,Science Kit,NaN,left_only
38,P039,Drawing Tablet Kids,NaN,left_only
39,P040,Kite,NaN,left_only


## 3. Different key names, and a self-merge
When the key columns have different names use `left_on` and `right_on`. `suffixes` renames columns that exist on both sides.

`staff.manager_id` points at another row's `staff_id`, so merging `staff` with itself attaches each person's manager.

In [5]:
managers = staff[["staff_id", "name", "role"]]

org = staff.merge(managers, left_on="manager_id", right_on="staff_id", how="left", suffixes=("", "_manager"))
org[["staff_id", "name", "role", "name_manager", "role_manager"]].head(6)

,staff_id,name,role,name_manager,role_manager
0,S01,Liam Menon,Chief Executive,NaN,NaN
1,S02,Lucas Schmidt,Head of Sales,Liam Menon,Chief Executive
2,S03,Diego Iyer,Head of Support,Liam Menon,Chief Executive
3,S04,Aarav Garcia,Head of Operations,Liam Menon,Chief Executive
4,S05,Emma Sharma,Sales Team Lead,Lucas Schmidt,Head of Sales
5,S06,Lea Weber,Sales Team Lead,Lucas Schmidt,Head of Sales


## 4. `validate`: state the relationship
Every merge assumes a relationship between the keys. `validate` makes pandas check it and raise an error if the data disagrees.

| `validate` | Meaning |
| --- | --- |
| `"one_to_one"` | keys unique on both sides |
| `"many_to_one"` | keys unique on the **right** |
| `"one_to_many"` | keys unique on the **left** |

Many orders belong to one customer, so `many_to_one` passes and `one_to_one` fails.

In [6]:
orders.merge(customers, on="customer_id", validate="many_to_one")
print("many_to_one: passed")

try:
    orders.merge(customers, on="customer_id", validate="one_to_one")
except pd.errors.MergeError as error:
    print("one_to_one : MergeError:", error)

many_to_one: passed
one_to_one : MergeError: Merge keys are not unique in left dataset; not a one-to-one merge
Duplicates in left:
 customer_id
      C0150
      C0181
      C0200
      C0033
      C0221 ...


## 5. Pitfall: fan-out
Merging a "one" table to a "many" table **repeats** the "one" rows. Any order-level number is then counted once per order line.

Below, every order has a shipping fee of 4.99. After merging with the order lines the fee total more than doubles.

In [7]:
order_fees = orders[["order_id"]].assign(shipping_fee=4.99)
joined = order_fees.merge(items, on="order_id")

print("rows: orders", len(order_fees), "-> after merge", len(joined))
print("true shipping total    :", round(order_fees["shipping_fee"].sum(), 2))
print("total after the merge  :", round(joined["shipping_fee"].sum(), 2))

rows: orders 2000 -> after merge 4439
true shipping total    : 9980.0
total after the merge  : 22150.61


**Fix:** bring both tables to the same grain first. Aggregate the lines to one row per order, then merge one-to-one and let `validate` guard it.

In [8]:
order_totals = items.groupby("order_id", as_index=False).agg(lines=("line_no", "size"), revenue=("revenue", "sum"))
per_order = order_fees.merge(order_totals, on="order_id", validate="one_to_one")

print("rows:", len(per_order), "| shipping total:", round(per_order["shipping_fee"].sum(), 2))
per_order.round(2).head(3)

rows: 2000 | shipping total: 9980.0


,order_id,shipping_fee,lines,revenue
0,10001,4.99,2,194.0
1,10002,4.99,2,74.0
2,10003,4.99,1,28.0


## 6. Keys that do not match
A merge compares keys **exactly**. Two frequent causes of silently missing matches:

**Different dtypes.** `10001` (number) is not `"10001"` (text). pandas refuses to merge a numeric key with a text key.

In [9]:
items_text_key = items.astype({"order_id": "str"})

try:
    orders.merge(items_text_key, on="order_id")
except ValueError as error:
    print("ValueError:", error)

fixed = orders.merge(items_text_key.astype({"order_id": "int64"}), on="order_id")
print("after converting the key:", len(fixed), "rows")

ValueError: You are trying to merge on int64 and str columns for key 'order_id'. If you wish to proceed you should use pd.concat
after converting the key: 4439 rows


**Different formatting.** Case and stray spaces make equal-looking keys unequal. `orders_raw.csv` contains lower-case customer ids. Normalise keys before merging, and use `indicator` to count what is still unmatched.

In [10]:
raw = pd.read_csv(DATA / "orders_raw.csv")


def match_report(frame):
    merged = frame.merge(customers[["customer_id", "country"]], on="customer_id", how="left", indicator=True)
    return merged["_merge"].value_counts().to_dict()


print("as loaded      :", match_report(raw))

raw["customer_id"] = raw["customer_id"].str.strip().str.upper()
print("after cleaning :", match_report(raw))

still_unmatched = raw.loc[~raw["customer_id"].isin(customers["customer_id"]), "customer_id"]
print("still unmatched:", still_unmatched.value_counts(dropna=False).to_dict())

as loaded      : {'both': 359, 'left_only': 42, 'right_only': 0}
after cleaning : {'both': 384, 'left_only': 17, 'right_only': 0}
still unmatched: {'C9999': 12, nan: 5}


The remaining rows are a different problem: an id that does not exist in the customer table, and missing ids. No formatting fix can repair those.

## 7. `join` and `concat`
- `df.join(other)` merges on the **index**. It is a shortcut for adding columns from a frame or Series indexed by the key.
- `pd.concat([a, b])` **stacks** frames: rows on top of each other by default, side by side with `axis=1`.

In [11]:
spend = (
    items.merge(orders[["order_id", "customer_id"]], on="order_id")
    .groupby("customer_id")["revenue"]
    .sum()
    .round(2)
    .rename("spend")
)

customers.set_index("customer_id")[["name", "country"]].join(spend).sort_values("spend", ascending=False).head(4)

,name,country,spend
customer_id,,,
C0181,Grace Roy,Canada,24370.90
C0033,Hugo Brown,Canada,23402.56
C0150,Felix Moreau,Spain,16682.85
C0171,Priya Iyer,USA,12644.67


In [12]:
first_half = orders[orders["order_date"] < "2024-07-01"]
second_half = orders[orders["order_date"] >= "2024-07-01"]

stacked = pd.concat([first_half, second_half], ignore_index=True)
print(len(first_half), "+", len(second_half), "=", len(stacked))

# keys= labels each piece, which is handy when combining files.
labelled = pd.concat([first_half, second_half], keys=["H1-2024", "later"], names=["batch", "row"])
labelled.groupby(level="batch").size()

541 + 1459 = 2000


batch
H1-2024     541
later      1459
dtype: int64

## 8. `merge_asof`: match on the nearest earlier key
A normal merge needs equal keys. `merge_asof` matches each left row with the **last right row at or before it**, which answers "what was the value at that time?".

Example: a product's list price changes over time. For each order line, find the price that was in force on the order date.
- Both frames must be **sorted** by the time key.
- `by=` restricts matches to the same product.

In [13]:
price_history = pd.DataFrame(
    {
        "product_id": ["P001", "P001", "P001"],
        "effective_from": pd.to_datetime(["2024-01-01", "2024-09-01", "2025-03-01"]),
        "list_price": [22.99, 24.99, 26.99],
    }
)
price_history["effective_from"] = price_history["effective_from"].astype(orders["order_date"].dtype)

mouse_lines = (
    items[items["product_id"] == "P001"]
    .merge(orders[["order_id", "order_date"]], on="order_id")
    .sort_values("order_date")
)

priced = pd.merge_asof(mouse_lines, price_history, left_on="order_date", right_on="effective_from", by="product_id")

priced.groupby("list_price").agg(
    lines=("order_id", "size"), first_order=("order_date", "min"), last_order=("order_date", "max")
)

,lines,first_order,last_order
list_price,,,
22.99,47,2024-01-03,2024-08-15
24.99,59,2024-09-01,2025-02-28
26.99,42,2025-03-01,2025-06-21


## 9. Cross join: every combination
`how="cross"` pairs every left row with every right row. It builds a complete grid, which you can then left-merge real data onto to reveal the combinations that never occurred.

In [14]:
channels = pd.DataFrame({"channel": sorted(orders["channel"].unique())})
grid = products[["product_id", "product_name"]].merge(channels, how="cross")

actual = (
    items.merge(orders[["order_id", "channel"]], on="order_id")
    .groupby(["product_id", "channel"], as_index=False)["quantity"]
    .sum()
)
coverage = grid.merge(actual, on=["product_id", "channel"], how="left")

print("grid:", len(products), "products x", len(channels), "channels =", len(grid), "rows")
print("combinations with no sales:", int(coverage["quantity"].isna().sum()))
coverage[coverage["quantity"].isna()].head(4)

grid: 40 products x 3 channels = 120 rows
combinations with no sales: 9


,product_id,product_name,channel,quantity
111,P038,Science Kit,mobile,NaN
112,P038,Science Kit,store,NaN
113,P038,Science Kit,web,NaN
114,P039,Drawing Tablet Kids,mobile,NaN


## A checklist for every merge
1. What is the **grain** of each table (what does one row mean)?
2. Is the key unique on the side where it should be? Use `validate`.
3. Do the key **dtypes and formats** agree?
4. Compare **row counts** before and after.
5. Use `indicator=True` to count unmatched rows on each side.

## Key takeaways
- `how` decides which unmatched rows survive.
- `indicator` and `validate` turn silent problems into visible ones.
- Aggregate to a common grain before merging to avoid fan-out.
- Clean keys first: same dtype, same case, no stray spaces.
- `merge_asof` joins on "the latest row at or before", not on equality.

## Practice
1. List the staff members who have no store orders.
2. Merge `orders` with `staff` to get revenue per sales rep. Which `validate` value fits?
3. Add each customer's first order date to `customers` with `join`.
4. Build a complete country × category grid and show revenue for each cell, with 0 where there were no sales.